# 修改模型后，怎样判断改进是否可靠？：参考答案

每道题先写预期结果再运行。练习中的 TODO 需要自行完成；能顺序运行不代表已完成练习。

In [1]:
from pathlib import Path
CASE_DIR = Path.cwd()
# 载入本例的可修改函数和数据，所有算法都在同目录 solution.py。
exec(compile((CASE_DIR / "solution.py").read_text(encoding="utf-8"), str(CASE_DIR / "solution.py"), "exec"))

{
  "baseline_accuracy": 0.8684027777777777,
  "improved_accuracy": 0.8909722222222223,
  "mean_accuracy_gain": 0.022569444444444444,
  "paired_95_percent_interval": [
    -0.0006944444444444445,
    0.04618055555555556
  ],
  "per_seed_gain": [
    0.030555555555555544,
    0.022222222222222227,
    0.02500000000000001,
    0.024999999999999994,
    0.016666666666666663,
    0.01944444444444446,
    0.02777777777777779,
    0.013888888888888881
  ],
  "mean_training_seconds": [
    0.0017870999945444055,
    0.0030810749958618544
  ],
  "interval_scope": "fixed trained models; iid test examples; not full training uncertainty",
  "elapsed_seconds": 1.213
}


## 1. 补全：配对方向
核对基线错误减新方案错误是改善方向；所有样本改善0.2时区间必须也是0.2。

In [2]:
ci, draws = paired_bootstrap(np.full(30, 0.2), repeats=100)
assert np.allclose(ci, [0.2, 0.2]) and np.allclose(draws, 0.2)
print(ci)

[0.2 0.2]


## 2. 修改：报告最差种子
计算提升最小的种子与训练成本比，不丢掉差结果。

In [3]:
worst = int(np.argmin(per_seed_gain))
ratio = float(np.mean(timings, axis=0)[1] / np.mean(timings, axis=0)[0])
assert per_seed_gain[worst] == per_seed_gain.min() and ratio > 0
print('最弱种子', seeds[worst], '改善', per_seed_gain[worst], '训练耗时比', ratio)

最弱种子 7 改善 0.013888888888888881 训练耗时比 1.7240641291856356


## 3. 迁移：有时间相关性时按区块抽样
把连续24个损失作为一个区块，先计算每块均值再 bootstrap；不把相邻时刻当完全独立。

In [4]:
rng = np.random.default_rng(3)
daily = rng.normal(0.1, 0.2, 20)
hourly = np.repeat(daily, 24) + rng.normal(0, 0.02, 480)
blocks = hourly.reshape(20, 24).mean(1)
ci, draws = paired_bootstrap(blocks, repeats=500)
assert len(blocks) == 20 and ci[0] <= ci[1]
print('以天为独立单位的区间', ci)

以天为独立单位的区间 [-0.02455592  0.18647717]
